In [86]:
import pandas as pd
import numpy as np
import pickle
pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)

In [87]:

def evaluate_portfolio(w, returns):
    daily_ret = returns @ w
    ann_return = daily_ret.mean() * 252
    ann_vol = daily_ret.std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    return ann_return, ann_vol, sharpe

In [88]:
def build_efficiency_table(
    results,
    returns_is,
    returns_oos,
    vol_levels
):
    rows = []

    # --- ESG portfolios ---
    for vol in vol_levels:
        # closest model-implied volatility
        vols = np.array([r["Volatility"] for r in results])
        idx = np.argmin(np.abs(vols - vol))
        w = results[idx]["Weights"]

        is_metrics  = evaluate_portfolio(w, returns_is)
        oos_metrics = evaluate_portfolio(w, returns_oos)

        rows.append({
            "Volatility Target": vol,
            "Return (IS)": is_metrics[0],
            "Return (OOS)": oos_metrics[0],
            "Volatility (OOS)": oos_metrics[1],
            "Sharpe (IS)": is_metrics[2],
            "Sharpe (OOS)": oos_metrics[2],
        })

 

    return pd.DataFrame(rows)


In [89]:
returns_oos = pd.read_parquet("data/returns_oos_2024.parquet")
returns_is_5y = pd.read_parquet("data/returns_5y.parquet")
returns_is_10y =  pd.read_parquet("data/returns_10y.parquet")

with open("results/optimized_weights_10y.pkl", "rb") as f:
    results_10y = pickle.load(f)

with open("results/optimized_weights_5y.pkl", "rb") as f:
    results_5y = pickle.load(f)



def get_closest_result(results, vol_target):
    vols = np.array([r["Volatility"] for r in results])
    idx = np.argmin(np.abs(vols - vol_target))
    return results[idx]

idxes = [0, 4, 9, 14]

rows = []

for idx in idxes:
    
    # In-sample (IS)
    is_5y = evaluate_portfolio(results_5y[idx]["Weights"], returns_is_5y)
    is_10y = evaluate_portfolio(results_10y[idx]["Weights"], returns_is_10y)

    # out-of-sample (OOS) (2024)
    oos_5y = evaluate_portfolio(results_5y[idx]["Weights"], returns_oos)
    oos_10y = evaluate_portfolio(results_10y[idx]["Weights"], returns_oos)

    vol_levels = [0.16, 0.20, 0.25, 0.30]

table_5y = build_efficiency_table(
    results=results_5y,
    returns_is=returns_is_5y,
    returns_oos=returns_oos,
    vol_levels=vol_levels
)

table_10y = build_efficiency_table(
    results=results_10y,
    returns_is=returns_is_10y,
    returns_oos=returns_oos,
    vol_levels=vol_levels
)


In [90]:
def build_benchmark_efficiency_table(w_is, w_oos, returns_is, returns_oos):
    is_metrics  = evaluate_portfolio(w_is, returns_is)
    oos_metrics = evaluate_portfolio(w_oos, returns_oos)

    return pd.DataFrame([{
        
        "Return (IS)": is_metrics[0],
        "Return (OOS)": oos_metrics[0],
        "Volatility (IS)": is_metrics[1],
        "Volatility (OOS)": oos_metrics[1],
        "Sharpe (IS)": is_metrics[2],
        "Sharpe (OOS)": oos_metrics[2],
    }])


In [91]:
w_bench = pd.read_csv("data/benchmark_weights.csv").set_index("Instrument")['BenchWeight']

In [92]:
bench_table_5y = build_benchmark_efficiency_table(
    w_is=w_bench,
    w_oos=w_bench,
    returns_is=returns_is_5y,
    returns_oos=returns_oos
)

bench_table_10y = build_benchmark_efficiency_table(
    w_is=w_bench,
    w_oos=w_bench,
    returns_is=returns_is_10y,
    returns_oos=returns_oos
)


In [93]:
table_5y

,Volatility Target,Return (IS),Return (OOS),Volatility (OOS),Sharpe (IS),Sharpe (OOS)
0,0.16,0.036180,0.127777,0.105039,0.225335,1.216465
1,0.20,-0.012250,-0.139839,0.172062,-0.060896,-0.812725
2,0.25,-0.101288,-0.332617,0.279974,-0.402419,-1.188029
3,0.30,-0.172680,-0.728812,0.425553,-0.572701,-1.712624


In [94]:
table_10y

,Volatility Target,Return (IS),Return (OOS),Volatility (OOS),Sharpe (IS),Sharpe (OOS)
0,0.16,0.041737,0.010543,0.137291,0.260236,0.076796
1,0.20,-0.000228,-0.207072,0.240900,-0.001133,-0.859574
2,0.25,-0.062990,-0.641336,0.398754,-0.251211,-1.608351
3,0.30,-0.078561,-1.043232,0.532634,-0.261313,-1.958627


In [95]:
bench_table_10y

,Return (IS),Return (OOS),Volatility (IS),Volatility (OOS),Sharpe (IS),Sharpe (OOS)
0,0.15626,0.14511,0.194557,0.116252,0.803157,1.248233


In [96]:
bench_table_5y


,Return (IS),Return (OOS),Volatility (IS),Volatility (OOS),Sharpe (IS),Sharpe (OOS)
0,0.185398,0.14511,0.228162,0.116252,0.812573,1.248233


In [97]:
table_5y_fmt = table_5y.round(2)
table_10y_fmt = table_10y.round(2)
bench_table_5y_fmt = bench_table_5y.round(2)
bench_table_10y_fmt = bench_table_10y.round(2)

latex_5y = table_5y_fmt.to_latex(
    index=False,
    float_format="%.2f",
    caption="In-sample (IS) and out-of-sample (OOS) performance of ESG-optimized portfolios (5-year estimation window).",
    label="tab:efficiency_5y",
    escape=False
)
latex_5y = latex_5y.replace(
    "\\begin{tabular}",
    "\\resizebox{1\\textwidth}{!}{%\n\\begin{tabular}"
).replace(
    "\\end{tabular}",
    "\\end{tabular}\n}"
)
latex_10y = table_10y_fmt.to_latex(
    index=False,
    float_format="%.2f",
    caption="In-sample (IS) and out-of-sample (OOS) performance of ESG-optimized portfolios (10-year estimation window).",
    label="tab:efficiency_10y",
    escape=False
)
latex_10y = latex_10y.replace(
    "\\begin{tabular}",
    "\\resizebox{1\\textwidth}{!}{%\n\\begin{tabular}"
).replace(
    "\\end{tabular}",
    "\\end{tabular}\n}"
)
latex_bench_5y = bench_table_5y_fmt.to_latex(
    index=False,
    float_format="%.2f",
    caption="In-sample (IS) and out-of-sample (OOS) performance of the benchmark portfolio (5-year estimation window).",
    label="tab:efficiency_benchmark_5y",
    escape=False
)
latex_bench_5y = latex_bench_5y.replace(
    "\\begin{tabular}",
    "\\resizebox{1\\textwidth}{!}{%\n\\begin{tabular}"
).replace(
    "\\end{tabular}",
    "\\end{tabular}\n}"
)

latex_bench_10y = bench_table_10y_fmt.to_latex(
    index=False,
    float_format="%.2f",
    caption="In-sample (IS) and out-of-sample (OOS) performance of the benchmark portfolio (10-year estimation window).",
    label="tab:efficiency_benchmark_10y",
    escape=False
)
latex_bench_10y = latex_bench_10y.replace(
    "\\begin{tabular}",
    "\\resizebox{1\\textwidth}{!}{%\n\\begin{tabular}"
).replace(
    "\\end{tabular}",
    "\\end{tabular}\n}"
)

with open("results/efficiency_5y.tex", "w") as f:
    f.write(latex_5y)

with open("results/efficiency_10y.tex", "w") as f:
    f.write(latex_10y)

with open("results/efficiency_benchmark_5y.tex", "w") as f:
    f.write(latex_bench_5y)

with open("results/efficiency_benchmark_10y.tex", "w") as f:
    f.write(latex_bench_10y)
